## Welcome to Week 6 Day 3: Context Engineering with MCP

People used to talk about Prompt Engineering, but now it's given way to a new Skill "Context Engineering".

Philipp Schmid of Google DeepMind wrote the seminal post about Context Engineering:

https://www.philschmid.de/context-engineering

For this week, we will put Context Engineering into practice, heavily using MCP servers.

1. Long-term memory: a knowledge graph the agent writes to and reads back
2. Web search: fresh information from the live web
3. Agentic RAG: a vector store the agent fills from its own research, then searches
4. Integrations: connecting to a live external service, with a local fallback

In [19]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio, create_static_tool_filter
from openai import AsyncOpenAI
import os
from pathlib import Path
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv(override=True)

True

### A quick note for Windows

Launching a local MCP server from a notebook hits one rough edge on Windows. The server writes its startup output to stderr, but inside a Windows Jupyter kernel that stream has no real file handle behind it, so the launch fails with an `io.UnsupportedOperation: fileno` error, while Mac and Linux are unaffected.

The fix is to send the server's stderr to the null device, so it always has somewhere real to write. We do that once in the next cell, which lets every cell below use `MCPServerStdio` exactly as the OpenAI Agents SDK documents it. On Mac and Linux it costs nothing beyond keeping the server's startup banner out of the notebook.

In [3]:
# On Windows, a stdio MCP server started from a Jupyter kernel writes to a stderr stream with no
# real file descriptor and crashes with io.UnsupportedOperation: fileno. We send the server's
# stderr to the null device so it always has somewhere real to write, which lets every cell below
# use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux are unaffected.
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

In [4]:
# this configuration is specific to non openai models
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("GOOGLE_API_KEY")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set - please head to the troubleshooting guide in the setup folder")

gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

gemini_model = OpenAIChatCompletionsModel(
    model="gemini-3.5-flash-lite",
    openai_client=gemini_client,
)

Google API Key exists and begins AQ.Ab8RN


## Part 1: Long-term memory

Our first context source is memory. This is the official knowledge graph server: it stores entities, observations about them, and the relationships between them, and keeps them on disk between runs. We point it at `memory/memory.json`, so you can open that file and read the graph it builds.

That gives an agent something it normally lacks: a memory that outlives the conversation. The agent writes facts as it learns them and reads them back later.

https://github.com/modelcontextprotocol/servers/tree/main/src/memory

In [5]:
memory_path = os.path.abspath("memory/memory.json")
memory_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-memory"], "env": {"MEMORY_FILE_PATH": memory_path}}

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as server:
    memory_tools = await server.list_tools()

memory_tools

[Tool(name='create_entities', title='Create Entities', description='Create multiple new entities in the knowledge graph', inputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'The type of the entity'}, 'observations': {'type': 'array', 'items': {'type': 'string'}, 'description': 'An array of observation contents associated with the entity'}}, 'required': ['name', 'entityType', 'observations']}}}, 'required': ['entities']}, outputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'The type of the entity'}, 'observations': {'

In [7]:
instructions = "You use your entity tools as a persistent memory to store and recall information about your conversations."
request = """My name's Ed. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. 
MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities.
"""
model = gemini_model

In [8]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Nice to meet you, Ed! It's great to have an LLM engineer here. 

Teaching a course on AI Agents—especially focusing on the Model Context Protocol (MCP)—sounds like fantastic work. MCP is a game-changer for standardizing how agents interact with external tools, resources, and prompt templates. 

I've made a note of who you are and what you're working on. How can I help you today?

In [9]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "My name's Ed. What do you know about me?")
    display(Markdown(result.final_output))

Hi Ed! Based on what I have stored in my memory:

* You are an **LLM engineer**.
* You are teaching a course about **AI Agents**.
* As part of your course, you teach the **MCP Protocol** (Model Context Protocol), which connects AI agents with tools, resources, and prompt templates. 

Let me know if there's anything else you'd like me to remember or update!

### Check out the trace

https://platform.openai.com/traces

## Part 2: Web search

A model only knows what it was trained on. Web search is the context source that keeps it current.

We use Tavily, a search API built for agents: it returns clean, ranked results an LLM can use directly. Tavily maintains its own MCP server, so connecting is a one-liner.

This needs a free API key:

1. Sign up at https://www.tavily.com
2. The free tier gives you 1,000 searches a month, with no credit card
3. Copy your API key (it starts with `tvly-`) and add it to your `.env` file:

`TAVILY_API_KEY=tvly-xxxx`

In [11]:
tavily_params = {"command": "npx", "args": ["-y", "tavily-mcp@latest"], "env": {"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")}}

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60) as server:
    tavily_tools = await server.list_tools()

tavily_tools

[Tool(name='tavily_search', title=None, description='Search the web for current information on any topic. Use for news, facts, or data beyond your knowledge cutoff. Returns snippets and source URLs.', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query'}, 'search_depth': {'type': 'string', 'enum': ['basic', 'advanced', 'fast', 'ultra-fast'], 'description': "The depth of the search. 'basic' for generic results, 'advanced' for more thorough search, 'fast' for optimized low latency with high relevance, 'ultra-fast' for prioritizing latency above all else", 'default': 'basic'}, 'topic': {'type': 'string', 'enum': ['general'], 'description': 'The category of the search. This will determine which of our agents will be used for the search', 'default': 'general'}, 'time_range': {'type': 'string', 'description': 'The time range back from the current date to include in the search results', 'enum': ['day', 'week', 'month', 'year']}, 'start_date':

Tavily's server offers several tools (search, extract, crawl, map, research). For this lab we only want plain web search, so we restrict the server to `tavily_search`. The OpenAI Agents SDK lets you hand an agent just the tools you choose with a static tool filter. Curating an agent's tools like this is itself context engineering.

In [12]:
instructions = "You search the web for information and briefly summarize the takeaways."
request = f"Please research the latest news on Amazon stock price and briefly summarize its outlook. For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = gemini_model
search_only = create_static_tool_filter(allowed_tool_names=["tavily_search"])

In [13]:
async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

As of mid-August 2026, Amazon (NASDAQ: AMZN) stock is trading in the **$265 range**, pulling back slightly from an all-time intraday high of around $290 reached earlier in the month following a strong second-quarter earnings report. 

### Key Drivers & Market Context:
* **Q2 Earnings & AWS/AI Momentum:** Amazon's recent earnings report beat expectations, fueled by robust performance and expanding margins in Amazon Web Services (AWS). Analysts continue to highlight AWS's growth and artificial intelligence (AI) integration as core pillars for the company's valuation re-rating.
* **Recent Volatility:** Despite the earnings beat, the stock has experienced minor profit-taking and consolidation—dropping roughly 2.5% over a 10-day span from its peak—amid broader headlines including founder share sales and ongoing regulatory/antitrust scrutiny.

### Wall Street & Technical Outlook:
* **Analyst Consensus:** Wall Street maintains a **Buy / Moderate Buy** consensus rating on AMZN. Major financial institutions have adjusted their price targets upward following the Q2 report, with consensus targets clustering around **$322 to $335**, suggesting double-digit upside potential from current trading levels. 
* **Technical Outlook:** Short-term technical indicators show signs of stabilization, with recent sessions flashing buy signals. Analysts view the current consolidation phase around the mid-$260s as a healthy retracement after hitting record highs, leaving room for continued upward momentum into the fall.

## Part 3: Agentic RAG

RAG, retrieval augmented generation, means giving a model relevant documents to ground its answer. The usual setup loads documents into a vector store up front. Agentic RAG turns that around: the agent builds the knowledge base itself, deciding what is worth keeping and storing it as it works.

We use the official Qdrant MCP server. Qdrant is a vector database, and the server exposes two tools: one stores a piece of text, the other finds the most relevant stored text for a query. It runs fully locally here. `QDRANT_LOCAL_PATH` keeps everything on disk with no separate database to run, and it embeds text with a local model, so there is no extra API key. The first store or search downloads that small embedding model, so it pauses once on first use.

https://github.com/qdrant/mcp-server-qdrant

We hand one agent both Tavily and Qdrant: it researches a topic on the web, stores what it learns, then answers from its own knowledge base.

In [14]:
vectordb_path = Path("memory/qdrant")
vectorstore_params = {
    "command": "uvx",
    "args": ["mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as server:
    vectorstore_tools = await server.list_tools()

vectorstore_tools

[Tool(name='qdrant-find', title=None, description='Look up memories in Qdrant. Use this tool when you need to: \n - Find memories by their content \n - Access memories for further analysis \n - Get some personal information about the user', inputSchema={'properties': {'query': {'description': 'What to search for', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='qdrant-store', title=None, description='Keep the memory for later use, when you are asked to remember something.', inputSchema={'properties': {'information': {'description': 'Text to store', 'title': 'Information', 'type': 'string'}, 'metadata': {'anyOf': [{'additionalProperties': True, 'type': 'object'}, {'type': 'null'}], 'default': None, 'description': 'Extra metadata stored along with memorised information. Any json is accepted.', 'title': 'Metadata'}}, 'required': ['information'], 'type': 'object'}, outpu

In [15]:
INSTRUCTIONS = """You research topics on the web and build up a knowledge base for later.
When you learn something worth keeping, store it in your knowledge base.
When you are asked what you know, search your knowledge base and answer from it."""

model = gemini_model

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as search_server:
    async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
        agent = Agent(name="researcher", instructions=INSTRUCTIONS, model=model, mcp_servers=[search_server, vector_server])
        with trace("research and store"):
            result = await Runner.run(agent, "Research the latest news on Nvidia and store the key facts in your knowledge base.", max_turns=20)
        display(Markdown(result.final_output))

I have researched the latest developments on Nvidia and successfully stored the key facts in my knowledge base. 

Here is a summary of the key findings that have been recorded:

1. **Next-Gen Networking & Infrastructure:** Nvidia introduced its next-generation Spectrum-X Ethernet networking platform designed to connect millions of GPUs for "gigascale" AI factories while lowering power consumption and operational costs. Strategic hardware integrations include partnerships with Cisco (e.g., the Cisco N9100 series data center switch built on Spectrum-X silicon).
2. **Product Roadmap:** 
   - **Blackwell Ultra** is ramping up quickly, accelerating revenue growth.
   - **Vera Rubin** (CPU/GPU platform) is on track for rollout in 2026, with a major partnership with OpenAI to deploy at least 10 gigawatts of Nvidia systems starting in the second half of 2026.
   - Follow-up architectures include **Rubin CPX** (late 2026), **Rubin Ultra** (2027), and **Feynman** (2028).
3. **China Market & Backlog:** Nvidia has expanded or resumed access to the Chinese market (including supplying powerful H200 processors to major tech giants like Alibaba, ByteDance, and Tencent), further bolstering an already massive data center order backlog.
4. **Major Partnerships:** Broad ecosystem commitments continue to expand through significant deals with companies and institutions such as OpenAI, the U.S. Department of Energy, Palantir, CrowdStrike, Uber, and Eli Lilly.

The agent below has only the knowledge base, no web search. Whatever it tells us about Nvidia, it is recalling from what the first agent stored. That is the retrieval half of RAG.

In [16]:
async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
    agent = Agent(name="researcher", instructions=INSTRUCTIONS, model=model, mcp_servers=[vector_server])
    with trace("retrieve"):
        result = await Runner.run(agent, "Based on your knowledge base, what's the latest on Nvidia?")
    display(Markdown(result.final_output))

Based on my knowledge base, here is the latest on Nvidia:

1. **Networking & Infrastructure**: Nvidia recently introduced the next-gen **Spectrum-X Ethernet networking platform** to connect millions of GPUs for gigascale AI factories while improving power efficiency and lowering operational costs. They have partnered with Cisco, which utilizes this technology via the Cisco N9100 series data center switch.
2. **Product Roadmap**: 
   * **Blackwell Ultra** is ramping up quickly, driving strong revenue acceleration.
   * The **Vera Rubin CPU/GPU combination platform** and architecture are on track for a 2026 launch (with major partners like OpenAI planning to deploy at least 10 gigawatts of Nvidia systems using Rubin starting in H2 2026).
   * Future generations include the **Rubin CPX** at the end of 2026, **Rubin Ultra** in 2027, and the **Feynman architecture** in 2028.
3. **China Market Expansion**: Nvidia has resumed or expanded access to the Chinese market (including shipping/taking orders for H200 processors), which has significantly boosted its massive data center order backlog—previously around $275 billion—driven by strong demand from companies like Alibaba, ByteDance, and Tencent.
4. **Major Partnerships**: Nvidia has secured significant enterprise, government, and tech collaborations with companies and entities like OpenAI, the US Department of Energy, Uber, Eli Lilly, Palantir, CrowdStrike, and Cisco to consume massive amounts of AI compute and GPUs.

### Check out the trace

https://platform.openai.com/traces

## Part 4: Integrations

The last context source is a live external service. Here that is market data from Massive (formerly Polygon.io), a popular financial data provider that publishes its own MCP server.

Setting this up is optional. Massive offers a free API key with no credit card, giving you their real end of day market data:

1. Sign up at https://www.massive.com
2. Create an API key
3. Add it to your `.env` file:

`MASSIVE_API_KEY=xxxx`

If you would rather not sign up, the next cell falls back to a local market server, the same kind of MCP server we built on Day 2, which serves simulated prices so the rest of the lab still works.

In [25]:
massive_api_key = os.getenv("MASSIVE_API_KEY")

if massive_api_key:
    market_params = {
        "command": "uvx",
        "args": ["--from", "git+https://github.com/massive-com/mcp_massive@v0.10.0", "--with", "mcp<2", "mcp_massive"],
        "env": {"MASSIVE_API_KEY": massive_api_key},
    }
else:
    market_params = {"command": "uv", "args": ["run", "-m", "backend.market_server"]}

async with MCPServerStdio(params=market_params, client_session_timeout_seconds=120) as server:
    market_tools = await server.list_tools()

market_tools

[Tool(name='search_endpoints', title=None, description='Search for market data API endpoints and built-in finance functions by natural language query. Use this FIRST to find the right endpoint before calling call_api. Covers stocks, options, forex, crypto, futures, indices, ETFs, and economic data. Pass market to pin results to a specific asset class when you already know it; omit it and the server will infer from the query. Use detail="more" to see query parameter docs needed for building call_api requests.', inputSchema={'additionalProperties': False, 'properties': {'query': {'description': 'Natural language search query for API endpoints', 'minLength': 1, 'title': 'Query', 'type': 'string'}, 'scope': {'anyOf': [{'enum': ['all', 'endpoints', 'functions'], 'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Search scope: "endpoints" for API only, "functions" for local functions only, or "all"/omit for both', 'title': 'Scope'}, 'max_results': {'anyOf': [{'maximum': 2

In [27]:
instructions = "You answer questions about the stock market."
request = "What was the most recent price that Apple (AAPL) traded at?"
model = gemini_model

async with MCPServerStdio(params=market_params, client_session_timeout_seconds=120) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

According to the most recent trading data available, Apple (AAPL) traded at a closing price of **$253.79** on its last active trading session (March 31, 2026). 

Summary of that session's pricing:
* **Open:** $247.91
* **High:** $255.48
* **Low:** $247.10
* **Close:** $253.79

## That's it for today

Four MCP servers, four kinds of context: long-term memory, web search, a knowledge base the agent builds itself, and a live integration. Choosing which sources an agent needs, and wiring them in, is much of what context engineering is in practice.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Browse an MCP marketplace like glama.ai/mcp or smithery.ai and add another context source to this notebook, using one of today's four patterns.
            </span>
        </td>
    </tr>
</table>